In [1]:
%load_ext autoreload
%autoreload 2

import os

print(f"Workdir: {os.getcwd()}")


Workdir: /Users/nscyclone/Projects/JiraEstimator


In [5]:
from prepare_data import prepare_data as run_preparation
from split_data import split_data as run_splitting
from extract_embeddings import main as run_embedding_extraction
from train_catboost_estimate import main as train_estimate_model
from train_catboost_risk import main as train_risk_model

print("Starting pipeline")

print("\n[1/5] Preparing raw data")
run_preparation()

print("\n[2/5] Splitting data into TVT buckets")
run_splitting()

print("\n[3/5] Generating embeddings for train, validation and test sets")
run_embedding_extraction()

print("\n[4/5] Training CatBoostRegressor for estimate prediction")
train_estimate_model()

print("\n[5/5] Training CatBoostClassifier for risk prediction")
train_risk_model()

print("Done")


Starting pipeline

[1/5] Preparing raw data
Reading data from data/seed.csv
Imported 100000 rows
Rows left after filtering: 87418
Writing to data/dataset.csv
Saved to data/dataset.csv
Risk class distribution is:
risk_level
0    50112
1    19403
2    17903
Name: count, dtype: int64

[2/5] Splitting data into TVT buckets
Reading data from data/dataset.csv
Imported 87418 rows
Saved the training (69934 rows), validation (8742 rows) and test (8742 rows) datasets to data/train.csv, data/val.csv and data/test.csv

[3/5] Generating embeddings for train, validation and test sets
Using device: mps
Reading a dataset from data/train.csv


Processing train.csv: 100%|██████████| 2186/2186 [29:33<00:00,  1.23it/s]   


Saved embeddings matrix shape: (69934, 768)
Using device: mps
Reading a dataset from data/train.csv


Processing val.csv: 100%|██████████| 274/274 [03:00<00:00,  1.52it/s]


Saved embeddings matrix shape: (8742, 768)
Using device: mps
Reading a dataset from data/train.csv


Processing test.csv: 100%|██████████| 274/274 [03:07<00:00,  1.46it/s]


Saved embeddings matrix shape: (8742, 768)
Done.

[5/5] Training CatBoostClassifier for risk prediction
Loading embeddings from embeddings
Loaded features matrix. Train: (69934, 771), Val: (8742, 771), Test: (8742, 771)
CatBoost training
0:	learn: 1.0978345	test: 1.0978755	best: 1.0978755 (0)	total: 199ms	remaining: 6m 38s
1:	learn: 1.0970639	test: 1.0971447	best: 1.0971447 (1)	total: 310ms	remaining: 5m 10s
2:	learn: 1.0964191	test: 1.0965104	best: 1.0965104 (2)	total: 462ms	remaining: 5m 7s
3:	learn: 1.0958076	test: 1.0960093	best: 1.0960093 (3)	total: 620ms	remaining: 5m 9s
4:	learn: 1.0952108	test: 1.0954193	best: 1.0954193 (4)	total: 748ms	remaining: 4m 58s
5:	learn: 1.0946045	test: 1.0948711	best: 1.0948711 (5)	total: 895ms	remaining: 4m 57s
6:	learn: 1.0940277	test: 1.0943543	best: 1.0943543 (6)	total: 1.03s	remaining: 4m 52s
7:	learn: 1.0934671	test: 1.0939094	best: 1.0939094 (7)	total: 1.17s	remaining: 4m 50s
8:	learn: 1.0929715	test: 1.0934585	best: 1.0934585 (8)	total: 1.31s

In [3]:
from catboost import CatBoostRegressor
from train_catboost_estimate import evaluate, load_data
from config import CONFIG

_, _, _, _, X_test, y_test_raw = load_data()

model_path = CONFIG.get('catboost_estimate_model_save_path', 'models/catboost_estimate_model.cbm')
model = CatBoostRegressor().load_model(model_path)

print("Evaluating CBR estimate prediction on a test set")
evaluate(model, X_test, y_test_raw)


Loading embeddings from embeddings
Filtering train: keeping logged days between 0.031 and 12.090 FTE
Dropped 1152 outliers from train
Loaded features matrix. Train: (68782, 771), Val: (8742, 771), Test: (8742, 771)
Evaluating CBR estimate prediction on a test set
Evaluating model on test data (Target: Actual Logged Days)
CatBoost regression evaluation results (Logged Days in FTE):
  MAE:1.256 FTE
  RMSE: 2.564 FTE
  R² Score: 0.1184
  Median errors: 0.642 FTE
  95% of errors are below: 4.616 FTE
  Max errors: 55.621 FTE

Accuracy (Logged Days vs Predicted):
  ≤ 0.25 FTE (2h): 20.37% задач
  ≤ 0.50 FTE (4h): 40.45% задач
  ≤ 1.00 FTE: 66.86% задач
  > 2.00 FTE: 14.69% задач
